
# NDX Garden — Workbench
### Module 2: The Machine Learning Workflow — Hands-On

The lesson covered EDA, preprocessing, and the ERM-vs-true-risk gap conceptually. Here you'll work with a deliberately messy dataset — missing values, mixed types, unscaled columns, class imbalance — and run the full pipeline yourself, including reproducing the "fit on the whole dataset" mistake so you can see its effect directly.

Task cells are marked `# YOUR CODE HERE`. Full solutions are at the bottom to self-check against.

**Note:** this notebook is a supplementary practice space, separate from your progress in the main NDX Garden lesson. Nothing here is graded or tracked.



## Setup: a deliberately messy dataset

Generated below, on purpose messy in the ways real data usually is:
- Missing values in `income` and `age`
- A categorical column (`region`) that needs encoding
- Columns on wildly different numeric scales (`income` in the tens of thousands, `age` in the tens)
- Class imbalance in the target (`churned`) — most customers did **not** churn


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(0)
n = 400

age = np.random.normal(40, 12, n)
income = np.random.normal(55000, 20000, n)
region = np.random.choice(["North", "South", "East", "West"], n, p=[0.4, 0.3, 0.2, 0.1])

# true churn signal: younger + lower income skews toward churn, plus noise
churn_score = (-0.05 * age) + (-0.00003 * income) + np.random.normal(0, 0.8, n)
churned = (churn_score > np.percentile(churn_score, 80)).astype(int)  # ~20% churn rate, imbalanced

df = pd.DataFrame({"age": age, "income": income, "region": region, "churned": churned})

# introduce missingness
missing_age_idx = np.random.choice(df.index, size=30, replace=False)
missing_income_idx = np.random.choice(df.index, size=25, replace=False)
df.loc[missing_age_idx, "age"] = np.nan
df.loc[missing_income_idx, "income"] = np.nan

df.head()



## Exercise 1 — EDA: find the problems before touching them

1. Check how many missing values are in each column (`df.isna().sum()`).
2. Check the class balance of `churned` (`df["churned"].value_counts(normalize=True)`).
3. Plot a histogram of `income` and `age` — do you see any outliers?


In [ ]:

# YOUR CODE HERE



## Exercise 2 — Preprocess: the right way vs. the wrong way

First, split into train/test. **Then**, and only then, preprocess — this ordering is the entire point of the exercise.

1. Split `df` into train/test (80/20, `random_state=1`, `stratify` on `churned` since it's imbalanced).
2. **The wrong way:** fit a `StandardScaler` on `age`/`income` using the **full, combined** train+test data, then transform both. This is the mistake the lesson's pitfall callout warns about.
3. **The right way:** fit the same scaler on **training data only**, then transform train and test separately.
4. Print the fitted scaler's `.mean_` from both approaches — they should differ, since one saw the test set's distribution and one didn't. That difference **is** the leak, made concrete.
5. Handle the missing values (impute) and encode `region` (one-hot) — the right way, fit only on training data.


In [ ]:

# YOUR CODE HERE
X = df.drop(columns="churned")
y = df["churned"]

X_train, X_test, y_train, y_test = None, None, None, None

# WRONG: fit scaler on combined train+test
scaler_wrong = None

# RIGHT: fit scaler on train only
scaler_right = None

# compare scaler_wrong.mean_ vs scaler_right.mean_



## Exercise 3 — Full pipeline, then compare train vs. test error

1. Build a `ColumnTransformer` that imputes + scales the numeric columns (`age`, `income`) and one-hot-encodes `region` — fit only on training data (use an `sklearn.pipeline.Pipeline` so this happens automatically and correctly).
2. Fit a `LogisticRegression` on the transformed training data.
3. Report accuracy on **both** train and test sets.
4. Is train accuracy noticeably higher than test accuracy? This is a live version of the ERM-vs-true-risk gap: training accuracy is measured on the exact data the model was optimized against, so it's an optimistic estimate — test accuracy is the more honest one.


In [ ]:

# YOUR CODE HERE
numeric_features = ["age", "income"]
categorical_features = ["region"]

preprocessor = None  # ColumnTransformer: SimpleImputer+StandardScaler for numeric, OneHotEncoder for categorical

pipeline = None  # Pipeline([("preprocess", preprocessor), ("model", LogisticRegression())])

# fit on X_train/y_train, then print train accuracy and test accuracy



---
## Solutions

Try every exercise yourself first. These are full worked solutions for self-checking.


### Solution 1

In [ ]:

print("Missing values per column:")
print(df.isna().sum())
print()
print("Class balance (churned):")
print(df["churned"].value_counts(normalize=True))
print()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(df["income"].dropna(), bins=25)
axes[0].set_title("Income distribution")
axes[1].hist(df["age"].dropna(), bins=25)
axes[1].set_title("Age distribution")
plt.tight_layout()
plt.show()


### Solution 2

In [ ]:

X = df.drop(columns="churned")
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

num_cols = ["age", "income"]

# WRONG — fit on combined train+test (simulate by concatenating first)
X_combined_num = pd.concat([X_train[num_cols], X_test[num_cols]])
imputer_wrong = SimpleImputer(strategy="mean").fit(X_combined_num)
X_combined_imputed = imputer_wrong.transform(X_combined_num)
scaler_wrong = StandardScaler().fit(X_combined_imputed)

# RIGHT — fit on train only
imputer_right = SimpleImputer(strategy="mean").fit(X_train[num_cols])
X_train_imputed = imputer_right.transform(X_train[num_cols])
scaler_right = StandardScaler().fit(X_train_imputed)

print("Wrong (fit on train+test) scaler means:", scaler_wrong.mean_)
print("Right (fit on train only) scaler means:", scaler_right.mean_)
print()
print("These differ because the 'wrong' scaler's mean/std were computed partly from data")
print("the model should never have been allowed to see before evaluation — this is the leak,")
print("made concrete as an actual numeric difference rather than an abstract warning.")


### Solution 3

In [ ]:

numeric_features = ["age", "income"]
categorical_features = ["region"]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="mean")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression()),
])

pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, pipeline.predict(X_train))
test_acc = accuracy_score(y_test, pipeline.predict(X_test))

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")
print()
if train_acc > test_acc:
    print(f"Train accuracy is {train_acc - test_acc:.3f} higher than test — this gap is expected: "
          "the model was optimized against the training data specifically, so its training performance "
          "is a rosier (and less trustworthy) estimate than its test performance.")
else:
    print(f"Test accuracy came out {test_acc - train_acc:.3f} higher than train here. With a simple linear "
          "model, a modest dataset, and one particular train/test split, this can happen by chance — the "
          "gap isn't always in the 'expected' direction on any single split. This is itself a useful lesson: "
          "a single train/test split gives one noisy estimate, which is exactly why Module 5 introduces "
          "cross-validation instead of trusting one split alone.")



---
*NDX Garden — Module 2 Workbench. Back to [ndx.io](https://ndx.io) · Next: [Module 3 Workbench](#)*
